# ToolCall-200M — final 4B-token pretraining

This notebook qualifies and then repeatedly resumes the frozen 200.2M-parameter run on Kaggle **T4 x2**. Attach `vaibhavmugulavalli/toolcall-4b-pretraining-corpus-v1`, enable Internet, and add the `HF_TOKEN` and `WANDB_API_KEY` Kaggle Secrets before selecting `RUN_MODE = "train"`.

In [ ]:
# Edit only this cell. Run interactively with 'qualify' first.
RUN_MODE = "qualify"  # 'qualify' or 'train'
HF_REPO_ID = "YOUR_HF_USERNAME/ToolCall-200M-checkpoints"
WANDB_ENTITY = ""  # Leave empty for your default W&B account
GITHUB_REPO = "https://github.com/VaibhavMugulavalli/ToolCall-200M.git"
GITHUB_REF = "main"
DATA_ROOT = "/kaggle/input/toolcall-4b-pretraining-corpus-v1"
PROJECT_ROOT = "/kaggle/working/ToolCall-200M"
RUN_ROOT = "/kaggle/working/toolcall-200m-run"
assert RUN_MODE in {"qualify", "train"}
if RUN_MODE == "train":
    assert not HF_REPO_ID.startswith("YOUR_"), "Set the exact Hugging Face repo ID"

In [ ]:
import os, subprocess, sys
from pathlib import Path

project = Path(PROJECT_ROOT)
if not project.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_REF, GITHUB_REPO, str(project)], check=True)
os.chdir(project)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "training/final_pretraining/requirements.txt"], check=True)
print(subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__, torch.version.cuda)"], check=True, capture_output=True, text=True).stdout)

In [ ]:
# GPU and dataset gate. The trainer also rechecks these inside each rank.
gpu_rows = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True, capture_output=True, text=True).stdout.strip().splitlines()
print("GPUs:", gpu_rows)
assert len(gpu_rows) == 2, f"Expected two GPUs, found {len(gpu_rows)}"
assert all("T4" in row for row in gpu_rows), f"Expected T4 x2, got {gpu_rows}"
assert Path(DATA_ROOT).is_dir(), f"Attach the Kaggle dataset: {DATA_ROOT}"

In [ ]:
# Read-only corpus, tokenizer, parameter-count, and 4B token-plan validation.
subprocess.run([sys.executable, "-m", "training.final_pretraining.preflight", "--data-root", DATA_ROOT], check=True)

In [ ]:
# Load secrets only for the real run; they are never printed.
if RUN_MODE == "train":
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    os.environ["HF_REPO_ID"] = HF_REPO_ID
    if WANDB_ENTITY:
        os.environ["WANDB_ENTITY"] = WANDB_ENTITY
    from huggingface_hub import HfApi
    info = HfApi(token=os.environ["HF_TOKEN"]).repo_info(HF_REPO_ID, repo_type="model")
    print(f"Hugging Face checkpoint repo is reachable: {info.id}")

## Launch

`qualify` is disposable and must print `QUALIFICATION PASSED`. Then change the first cell to `train`, save a new Kaggle version, and schedule that version. A train invocation pauses at 675 minutes, uploads `resume/latest`, and exits normally. The printed W&B URL is the live graph dashboard.

In [ ]:
import time
base = ["deepspeed", "--num_gpus", "2", "-m", "training.final_pretraining.train", "--data-root", DATA_ROOT]
if RUN_MODE == "qualify":
    qualification_root = f"/kaggle/working/toolcall-200m-qualification-{int(time.time())}"
    command = base + ["--run-dir", qualification_root, "--qualification-steps", "2"]
else:
    command = base + ["--run-dir", RUN_ROOT, "--hub-repo-id", HF_REPO_ID, "--resume", "auto"]
print("Launching:", " ".join(command[:8]), "...")
subprocess.run(command, check=True)

In [ ]:
# The summary remains useful in the interactive notebook after a clean pause.
summary_path = Path(RUN_ROOT) / "summary.json"
if summary_path.is_file():
    print(summary_path.read_text())
else:
    print("Qualification finished; no durable training run was changed.")